In [1]:
!git clone https://github.com/PonChick3n/myLLMs

Cloning into 'myLLMs'...
remote: Enumerating objects: 68, done.
remote: Counting objects: 100% (68/68), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 68 (delta 36), reused 48 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (68/68), 198.47 KiB | 19.85 MiB/s, done.
Resolving deltas: 100% (36/36), done.


In [2]:
!wget -O dataset.zip "https://dataverse.pushdom.ru/api/access/dataset/:persistentId/?persistentId=doi:10.31860/openlit-2023.8-C005"
!unzip dataset.zip
!mkdir texts
!unzip texts.zip -d texts

--2026-03-20 17:26:24--  https://dataverse.pushdom.ru/api/access/dataset/:persistentId/?persistentId=doi:10.31860/openlit-2023.8-C005
Resolving dataverse.pushdom.ru (dataverse.pushdom.ru)... 81.29.132.120
Connecting to dataverse.pushdom.ru (dataverse.pushdom.ru)|81.29.132.120|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: unspecified [application/zip]
Saving to: ‘dataset.zip’

dataset.zip             [     <=>            ]   1006K   967KB/s    in 1.0s    

2026-03-20 17:26:27 (967 KB/s) - ‘dataset.zip’ saved [1030191]

Archive:  dataset.zip
  inflating: bibliography.tab        
  inflating: metadata.tab            
  inflating: readme.md               
  inflating: readme.pdf              
   creating: tests/
  inflating: tests/test_integrity.py  
  inflating: tests/tests_readme.md   
  inflating: texts.zip               
  inflating: MANIFEST.TXT            
Archive:  texts.zip
  inflating: texts/A1.txt            
  inflating: texts/A2.txt            
  inf

In [3]:
import glob

all_text = []
for file_path in glob.glob('./texts/*.txt'):
    file = open(file_path, 'r', encoding='utf8')
    all_text.append(file.read())

all_text = '\n\n\n'.join(all_text)

In [4]:
%cd myLLMs

/content/myLLMs


In [6]:
import torch
from torch.utils.data import DataLoader
from getdata import GetData
from BPE import BPE
from GPT1 import GPT

In [7]:
tokenizer = BPE.load('data/tokenizer.dill')
token_ids = tokenizer.encode(all_text)

Объект загружен из data/tokenizer.dill


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [9]:
n = int(0.7 * len(token_ids))
train_token_ids = token_ids[:n]
valid_token_ids = token_ids[n:]

train_dataset = GetData(data=train_token_ids, seq_len=20, device=device)
train_loader = DataLoader(train_dataset, batch_size=32)

valid_dataset = GetData(data=valid_token_ids, seq_len=20, device=device)
valid_loader = DataLoader(valid_dataset, batch_size=32)

In [ ]:
gpt = GPT(vocab_size=2000, max_seq_len=40, emb_size=8, num_heads=4, head_size=8, num_layers=4, dropout=0.1, device=device)
gpt.fit(train_loader, valid_loader, num_epoch=10, learning_rate=0.001)

In [ ]:
torch.save(gpt.state_dict(), "data/weights.pt")

In [ ]:
print(tokenizer.decode(valid_token_ids[:100]))

 будет этого никак.
О, доля человека злая!
Ах, отчего я не табак!..


Вам Музы, милые старушки,
Колпак связали в добрый час.
И, прицепив к нему гремушки,
Сам Феб надел его на вас.
Хотелось в том же мне уборе
Пред вами нынче щегольнуть
И в откровенном разговоре


In [ ]:
x = torch.tensor(valid_token_ids[:30], dtype=torch.long, device=gpt.device).unsqueeze(0)
generated_tokens = gpt.generate(x, 70, do_sample=True)
generated_text = tokenizer.decode(generated_tokens[0].detach().cpu().tolist())

print(generated_text)

 будет этого никак.
О, доля человека злая!
Ах, отчего я не табак!..


Вам Му
Дil разед,
ТвратПre ец мучалие —
В жеымть,
Некры стра слезы, слачил егостомлестпинас!

 любраведмы
Ис восмериблхи расz те зна, скиит
Д очи нихял;чноais , оскестидямкрывалстал вы ли!
В на
